In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

print("Connected")

Connected


In [2]:
orders = pd.read_sql("SELECT * FROM orders", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
order_payments = pd.read_sql("SELECT * FROM order_payments", engine)
customers = pd.read_sql("SELECT customer_id, customer_state FROM customers", engine)
sellers = pd.read_sql("SELECT seller_id, seller_state FROM sellers", engine)

items_agg = order_items.sort_values("order_item_id").groupby("order_id").agg(
    num_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    seller_id=("seller_id", "first"),
    num_unique_sellers=("seller_id", "nunique")
).reset_index()

items_agg = items_agg.merge(sellers, on="seller_id", how="left")

payments_agg = order_payments.groupby("order_id").agg(
    num_payments=("payment_sequential", "count"),
    total_payment_value=("payment_value", "sum")
).reset_index()

ml_table = orders.merge(items_agg, on="order_id", how="left")
ml_table = ml_table.merge(payments_agg, on="order_id", how="left")
ml_table = ml_table.merge(customers, on="customer_id", how="left")

print(ml_table.shape)
print("Orders with multiple sellers:", (ml_table["num_unique_sellers"] > 1).sum())
ml_table[["order_id", "customer_state", "seller_state", "num_unique_sellers"]].head()

(99441, 15)
Orders with multiple sellers: 1278


,order_id,customer_state,seller_state,num_unique_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,RJ,SP,1.0
1,00018f77f2f0320c557190d7a144bdd3,SP,SP,1.0
2,000229ec398224ef6ca0657da4fc703e,MG,MG,1.0
3,00024acbcdf0a6daa1e931b038114c75,SP,SP,1.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,SP,PR,1.0


In [3]:
ml_table["same_state"] = (ml_table["customer_state"] == ml_table["seller_state"]).astype(int)

print(ml_table["same_state"].value_counts(normalize=True))

same_state
0    0.643216
1    0.356784
Name: proportion, dtype: float64


In [4]:
temp = ml_table.dropna(subset=["order_delivered_customer_date"]).copy()
temp["is_late_temp"] = (temp["order_delivered_customer_date"] > temp["order_estimated_delivery_date"]).astype(int)

print(temp.groupby("same_state")["is_late_temp"].mean())

same_state
0    0.092662
1    0.060600
Name: is_late_temp, dtype: float64


In [6]:
ml_table.to_csv("ml_table_step1.csv", index=False)
print("Saved:", ml_table.shape)

Saved: (99441, 16)
